# Grading

The parts of grading that code can do, the same way for the agent and the baseline:

- **Ending.** the packet ends as the answer key expects.
- **Clarification.** a CLARIFY names exactly the scope parts the key lists; asking when
  the request was complete fails the ending.
- **Evidence.** every row a FINDING cites was returned by a tool in that run (pointer
  tokens are already checked when the packet is built).
- **No typed numbers.** the stray-digit validator finds nothing.
- **Budget.** the agent stays within 8 turns and 8 calls (the baseline has no limit).

Cases that expect CLARIFY form their own slice (language understanding) and stay out of
the paired comparison. The human part of grading (`docs/rubric.md`) judges what the
packet says; `packet_text` renders a packet for it.

In [ ]:
#| default_exp grading

In [ ]:
#| export
from collections.abc import Iterable
from dataclasses import dataclass

from agentic_forecast_review.generator import (
    Snapshot,
)
from agentic_forecast_review.schemas import (
    MAX_CALLS,
    MAX_TURNS,
    Case,
    ManifestEntry,
    Packet,
    TerminalState,
)
from agentic_forecast_review.validator import (
    known_identifiers,
    packet_stray_digits,
)

## One packet

In [ ]:
#| export
@dataclass(frozen=True)
class AutoGrade:
    case_id: str
    runner: str
    planted: str
    ambiguous: bool
    ending_ok: bool
    clarify_ok: bool
    rows_ok: bool
    digits_ok: bool
    budget_ok: bool
    calls: int
    usd: float
    seconds: float

    @property
    def passed(self) -> bool:
        """Every code-graded check passes."""
        return (
            self.ending_ok
            and self.clarify_ok
            and self.rows_ok
            and self.digits_ok
            and self.budget_ok
        )


def auto_grade(
    packet: Packet, case: Case, snapshot: Snapshot, entry: ManifestEntry
) -> AutoGrade:
    """Grade one packet against the answer key, by code."""
    expects_clarify = entry.expected_terminal is TerminalState.CLARIFY
    clarify_ok = not expects_clarify or (
        packet.clarification is not None
        and packet.clarification.missing == entry.expected_missing
    )
    returned = {
        row_id
        for step in packet.trace
        if step.result is not None
        for row_id in step.result.row_ids
    }
    cited = {row_id for claim in packet.claims for row_id in claim.row_refs}
    stray = packet_stray_digits(packet, known_identifiers(case, snapshot, packet))
    return AutoGrade(
        case_id=case.case_id,
        runner=packet.runner,
        planted=entry.planted.value,
        ambiguous=expects_clarify,
        ending_ok=packet.terminal_state is entry.expected_terminal,
        clarify_ok=clarify_ok,
        rows_ok=cited <= returned,
        digits_ok=not stray,
        budget_ok=packet.runner == "baseline"
        or (packet.cost.turns <= MAX_TURNS and packet.cost.calls <= MAX_CALLS),
        calls=packet.cost.calls,
        usd=packet.cost.usd,
        seconds=packet.cost.seconds,
    )

## Counts, by slice

Results are reported as `k/n` counts, never as bare percentages.

In [ ]:
#| export
def summary(grades: Iterable[AutoGrade]) -> dict[str, str]:
    """``k/n`` for the ending and for all code checks, per slice, plus resources."""
    grades = list(grades)
    out: dict[str, str] = {}
    for name, keep in (("paired", False), ("ambiguous", True)):
        part = [grade for grade in grades if grade.ambiguous is keep]
        if part:
            out[f"{name} ending"] = f"{sum(g.ending_ok for g in part)}/{len(part)}"
            out[f"{name} all checks"] = f"{sum(g.passed for g in part)}/{len(part)}"
    if grades:
        out["mean calls"] = f"{sum(g.calls for g in grades) / len(grades):.1f}"
        out["total usd"] = f"{sum(g.usd for g in grades):.4f}"
    return out

## A packet as text, for the human grader

The request, the ending, every claim with its numbers filled in, what is not
established, and the steps taken (tool, reason). The answer key is not shown.

In [ ]:
#| export
def packet_text(packet: Packet, case: Case) -> str:
    """A readable Markdown rendering of one packet."""
    lines = [
        f"**Request:** {case.request_text}",
        "",
        f"**Ending:** {packet.terminal_state.value}",
    ]
    if packet.clarification is not None:
        lines.append(
            f"**Asks about:** {', '.join(sorted(packet.clarification.missing))}"
        )
    lines += ["", "**Claims:**"]
    for claim, text in zip(packet.claims, packet.render(), strict=True):
        rows = f" (rows: {len(claim.row_refs)})" if claim.row_refs else ""
        lines.append(f"- {claim.label.value}: {text}{rows}")
    if not packet.claims:
        lines.append("- none")
    lines += ["", "**Not established:**"]
    lines += [f"- {text}" for text in packet.render_not_established()] or ["- none"]
    lines += ["", "**Steps:**"]
    for step in packet.trace:
        tool = step.call.args.tool if step.call is not None else "finish"
        failed = " (failed)" if step.result is not None and not step.result.ok else ""
        lines.append(f"{step.turn}. {tool}{failed}: {step.summary}")
    return "\n".join(lines)

## Examples

The baseline on both development families, graded by code.

In [ ]:
from agentic_forecast_review.baseline import run_baseline
from agentic_forecast_review.cases import DEV_FAMILIES, build_family

grades = []
for spec in DEV_FAMILIES:
    family = build_family(spec)
    settings = {s.case_id: s for s in family.run_settings}
    for case, entry in zip(family.cases, family.manifest, strict=True):
        snapshot = family.snapshots[case.snapshot_id]
        packet = run_baseline(case, snapshot, settings.get(case.case_id))
        grades.append(auto_grade(packet, case, snapshot, entry))
print(summary(grades))
assert summary(grades)["paired ending"] == "16/16"
assert summary(grades)["ambiguous all checks"] == "2/2"

One baseline packet as the grader sees it.

In [ ]:
family = build_family(DEV_FAMILIES[1])
case = family.cases[1]
print(packet_text(run_baseline(case, family.snapshots[case.snapshot_id]), case))

## The development runs

The agent ran twice on each of the 18 development cases, in two rounds. Round 1 showed
two weaknesses: findings cited call names instead of row IDs, and a request with no metric
was answered instead of clarified. Before round 2, `finish` began refusing rows no tool
returned, and the prompt spelt out when to clarify. Every saved run is replayed offline
before it is graded.

In [ ]:
from pathlib import Path

from agentic_forecast_review.agent import replay
from agentic_forecast_review.kill_gate import load_runs

results = {}
for name in ("round_1", "round_2"):
    agent = []
    for spec in DEV_FAMILIES:
        family = build_family(spec)
        settings = {s.case_id: s for s in family.run_settings}
        for case, entry in zip(family.cases, family.manifest, strict=True):
            snapshot = family.snapshots[case.snapshot_id]
            for repeat in ("r1", "r2"):
                (record,) = load_runs(Path("../runs/dev") / name / repeat, case.case_id)
                packet = replay(record, case, snapshot, settings.get(case.case_id))
                agent.append(auto_grade(packet, case, snapshot, entry))
    results[name] = summary(agent)
    print(f"agent {name}: {results[name]}")
print(f"baseline: {summary(grades)}")
assert results["round_1"]["paired ending"] == "32/32"